Импортирование библиотек

In [ ]:
import os
import math
import time
import random
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.7.1+cu118
torchvision: 0.22.1+cu118


In [9]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

RANDOM_STATE = 42
set_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

FAST_MODE = True
DATA_DIR = "./data"

BATCH_SIZE = 64  # ResNet тяжелее, чем SimpleCNN
EPOCHS_HEAD = 3 if FAST_MODE else 8
EPOCHS_FT   = 3 if FAST_MODE else 8

Device: cuda


In [10]:
from torchvision.datasets import STL10

Часть A (S10): датасет для классификации изображений, Вариант A: STL10 

In [11]:
# STL10 mean/std (рассчитаны на основе данных)
STL10_MEAN = (0.4467, 0.4398, 0.4066)
STL10_STD = (0.2603, 0.2566, 0.2713)

def load_stl10(data_dir: str = DATA_DIR):
    """
    Загрузка STL10 датасета.
    STL10 имеет специальные split'ы:
    - train: 5000 изображений
    - test: 8000 изображений
    - unlabeled: 100000 немаркированных изображений (не используем)
    """
    tf_train = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(STL10_MEAN, STL10_STD),
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(STL10_MEAN, STL10_STD),
    ])

    # Загружаем train и test split'ы
    ds_train_full = torchvision.datasets.STL10(
        root=data_dir, 
        split='train',  # 'train' для обучающей выборки
        download=True, 
        transform=tf_train
    )
    ds_test = torchvision.datasets.STL10(
        root=data_dir, 
        split='test',  # 'test' для тестовой выборки
        download=True, 
        transform=tf_test
    )

    return ds_train_full, ds_test

In [13]:
# Загружаем данные
ds_train_full, ds_test = load_stl10()

# Классы STL10 (совпадают с CIFAR-10)
class_names = [
    'airplane', 'bird', 'car', 'cat', 'deer',
    'dog', 'horse', 'monkey', 'ship', 'truck'
]
print("Train full:", len(ds_train_full))
print("Test:", len(ds_test))
print("Classes:", class_names)
print(f"Размер изображений: {ds_train_full[0][0].shape}")  # (3, 96, 96)

1.0%


KeyboardInterrupt: 

In [ ]:
def make_loaders(
    ds_train_full,
    ds_test,
    batch_size: int = BATCH_SIZE,
    val_ratio: float = 0.2,
    seed: int = RANDOM_STATE,
    fast_mode: bool = FAST_MODE,
):
    # train/val split
    n_total = len(ds_train_full)
    n_val = int(n_total * val_ratio)
    n_train = n_total - n_val

    ds_train, ds_val = random_split(
        ds_train_full,
        lengths=[n_train, n_val],
        generator=torch.Generator().manual_seed(seed),
    )

    # FAST_MODE: уменьшаем размер для скорости
    if fast_mode:
        rng = np.random.RandomState(seed)
        
        # Для STL10 изображения 96x96, поэтому можно взять меньше данных
        train_idx = rng.choice(len(ds_train), size=min(2000, len(ds_train)), replace=False)
        val_idx = rng.choice(len(ds_val), size=min(500, len(ds_val)), replace=False)
        test_idx = rng.choice(len(ds_test), size=min(500, len(ds_test)), replace=False)

        ds_train = Subset(ds_train, train_idx)
        ds_val = Subset(ds_val, val_idx)
        ds_test_small = Subset(ds_test, test_idx)
    else:
        ds_test_small = ds_test

    train_loader = DataLoader(
        ds_train, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=2, 
        pin_memory=True
    )
    val_loader = DataLoader(
        ds_val, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    test_loader = DataLoader(
        ds_test_small, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )

    return train_loader, val_loader, test_loader

# Создаем загрузчики
train_loader, val_loader, test_loader = make_loaders(ds_train_full, ds_test)

# Проверяем батч
batch = next(iter(train_loader))
x, y = batch
print("x:", x.shape, x.dtype)
print("y:", y.shape, y.dtype)
print(f"Минимальное значение в тензоре: {x.min():.3f}")
print(f"Максимальное значение в тензоре: {x.max():.3f}")

In [ ]:
def denorm_stl10(x: torch.Tensor) -> torch.Tensor:
    """
    Денормализация для STL10.
    x: (C,H,W) или (B,C,H,W) тензор в нормализованном виде
    """
    mean = torch.tensor(STL10_MEAN).view(1, 3, 1, 1) if x.dim() == 4 else torch.tensor(STL10_MEAN).view(3, 1, 1)
    std = torch.tensor(STL10_STD).view(1, 3, 1, 1) if x.dim() == 4 else torch.tensor(STL10_STD).view(3, 1, 1)
    
    # Перемещаем на тот же device, что и x
    mean = mean.to(x.device)
    std = std.to(x.device)
    
    return x * std + mean

@torch.no_grad()
def show_images(loader, n: int = 10) -> None:
    """
    Визуализация изображений из загрузчика.
    Для STL10 изображения 96x96, поэтому делаем чуть больше размер фигуры.
    """
    x, y = next(iter(loader))
    x = x[:n].cpu()
    y = y[:n].cpu()

    plt.figure(figsize=(16, 3))  # Увеличил размер для STL10 (96x96 вместо 32x32)
    for i in range(n):
        plt.subplot(1, n, i + 1)
        
        # Денормализуем и приводим к диапазону [0, 1]
        img = denorm_stl10(x[i]).clamp(0, 1).permute(1, 2, 0).numpy()
        
        plt.imshow(img)
        plt.title(class_names[y[i].item()], fontsize=10)
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

# Показываем изображения
print("Примеры изображений из STL10:")
show_images(train_loader, n=10)